In [ ]:
import pickle

import matplotlib
import matplotlib.pyplot as plt
import os
import mne
import numpy as np
import pandas as pd
import torch
import copy
import seaborn
from omegaconf import OmegaConf
import yaml
import argparse
from data.data_loader import load_eeg_data
import matplotlib.pylab as pylab
import gc
import quantus
from tqdm import tqdm
from captum.attr import GradientShap

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""


def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))


In [ ]:
def get_and_plot_topomap(explanations, aggregation_func, info, aggregation_func_name="mean", save_path="refactor_test/topoplots/", subject_idx=2, **kwargs):
    # as the outputs of the XAI approaches come in the form trials x channels x timepoints, some form of summary stat needs to be applied.
    # this function allows for passing only a single trial and computing the mean over the timepoint dim,
    # or passing all trials and aggregating over trial and timepoints dim
    # Note: If only a single trial is passed, it needs to be passed in the shape 1 x channel x timepoints
    fig, ax = plt.subplots(figsize=(6,6))
    data = explanations.squeeze()
    #ax.set_title(f"subject: {subject_idx}, aggregation: {aggregation_func_name}")
    mne.viz.plot_topomap(aggregation_func(data, **kwargs), info, axes=ax)

    dir_path = f"{save_path}/{aggregation_func_name}"
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)

    fig.savefig(f"{dir_path}/GradShap_{aggregation_func_name}_subject_{subject_idx}_topomap.png")
    return aggregation_func(data, **kwargs)

def get_and_plot_topomap_groupby(explanations, groupby_dict, aggregation_func, info, aggregation_func_name="mean", groupby_name="uncertainty", save_path="refactor_test/groupby/uncertainty/topoplots/", subject_idx=2, **kwargs):
    # as the outputs of the XAI approaches come in the form trials x channels x timepoints, some form of summary stat needs to be applied.
    # this function allows for passing only a single trial and computing the mean over the timepoint dim,
    # or passing all trials and aggregating over trial and timepoints dim
    # Note: If only a single trial is passed, it needs to be passed in the shape 1 x channel x timepoints
    for groupby_key in groupby_dict.keys():
        fig, ax = plt.subplots(figsize=(6,6))
        data = explanations.squeeze()
        ax.set_title(f"subject: {subject_idx}, aggregation: {aggregation_func_name}, group_by: {groupby_name}: {groupby_key}")
        mne.viz.plot_topomap(aggregation_func(data[groupby_dict[groupby_key]], **kwargs), info, axes=ax, show=False)

        dir_path = f"{save_path}/{aggregation_func_name}"
        if not os.path.exists(dir_path):
            os.makedirs(dir_path)

        fig.savefig(f"{dir_path}/GradShap_{aggregation_func_name}_{groupby_name}_{groupby_key}_subject_{subject_idx}_topoplot.png")


# potentially take shape of data as input(e.g. what if we have more than 60 channels?)
def get_and_plot_heatmap(explanations, aggregation_func, ch_names, aggregation_func_name="mean", save_path="refactor_test/heatmaps/" ,plot=True, subject_idx=2):
    # aggregates an entry/key of aggregate_dict with the fiven aggregation_func over the trial dimension
    fig, ax = plt.subplots(figsize=(7,5))
    ax.set_title(f"subject: {subject_idx}, aggregation: {aggregation_func_name}")
    hm = seaborn.heatmap(aggregation_func(explanations.reshape(-1,60,900),axis=0), yticklabels=ch_names, ax=ax)
    hm.set_yticklabels(hm.get_yticklabels(), fontsize=5)
    if plot:
        figure = hm.get_figure()
        dir_path = f"{save_path}/{aggregation_func_name}"
        if not os.path.exists(dir_path):
            os.makedirs(dir_path)

        figure.savefig(dir_path+"/"+"Gradshap"+"_"+aggregation_func_name+"_subject_"+str(subject_idx)+".png")
        figure.clear()

    return hm

#groupby multiple conditions can be done boolean mask and the boolean & operator
def get_and_plot_heatmap_groupby(explanations, groupby_dict, aggregation_func, ch_names, aggregation_func_name="mean", groupby_name="uncertainty", save_path="refactor_test/groupby/uncertainty/heatmaps/", plot=True, subject_idx=2):
    # the groupby_dict contains as keys the label of the condition we group by, and as values the indicies of the aggregate_dict for which this condition holdss
    for groupby_key in groupby_dict.keys():
        fig, ax = plt.subplots(figsize=(7,5))
        ax.set_title(f"subject: {subject_idx}, aggregation: {aggregation_func_name}, group_by: {groupby_name}: {groupby_key}")
        hm = seaborn.heatmap(aggregation_func(explanations.reshape(570,60,-1)[groupby_dict[groupby_key]],axis=0), yticklabels=ch_names, ax=ax)
        hm.set_yticklabels(hm.get_yticklabels(), fontsize=5)
        if plot:

            figure = hm.get_figure()
            dir_path = f"{save_path}/{aggregation_func_name}"
            if not os.path.exists(dir_path):
                os.makedirs(dir_path)

            figure.savefig(f"{dir_path}/{groupby_key}_{groupby_name}_{groupby_key}_subject_{subject_idx}.png")
            figure.clear()





In [ ]:
# award importance to a channel according to their importance ranking in a time point
def update_channel_points_linear(top_k_channel_names, channel_point_dict):
    for i, channel_name in enumerate(top_k_channel_names):
        channel_point_dict[channel_name] += 10-i
    return channel_point_dict

# award importance to a channel according to their importance ranking in a time point
def update_channel_points_same(top_k_channel_names, channel_point_dict):
    for i, channel_name in enumerate(top_k_channel_names):
        channel_point_dict[channel_name] += 1
    return channel_point_dict

def get_top_k_channels_per_timepoint(time_point_data, ch_names, k):
    # trial is of shape channels x timepoins
    # for each timepoint, find indices of top_k channels with highest activations 
    top_k_channels_indices = np.argsort(time_point_data)[::-1][:k]
    # get the channel names of the top_k channels 
    top_k_channels_names = np.take(ch_names, top_k_channels_indices)
    #print(top_k_channels_names)
    return top_k_channels_names


# adapt so you can see a running list of top channes per trial
def get_top_k_channels_per_trial(trial, ch_names, channel_point_dict , k, update_func):
    for time_point in range(trial.shape[1]):
        #print(f"time_point: {time_point}")
        top_k_channel_names = get_top_k_channels_per_timepoint(trial[:,time_point], ch_names, k)
        # update the points of the top_k channels of the timepoint according to the (linear, stricly monotone falling) update function
        # 
        # NOTE while assumption that all trials are equally important holds, assumption that all timepoints are equally important is very questionable. 
        # Maybe think about this part again
        channel_point_dict = update_func(top_k_channel_names, channel_point_dict)
    return channel_point_dict

def get_top_k_channels(explanations, ch_names, k, update_func):
    # get the channel importance for each channel according to the explanation method
    # a ranking approach is chosen s.t. each timepoint in a trial and each trial is equally important

    # init dict for storing points per channel

    channel_point_dicts_over_time = []

    channel_point_dict = {}
    for ch in ch_names:
        channel_point_dict[ch] = 0
    
    data = explanations.squeeze()
    # go through each trial in the dataset and assign points for each
    for trial in data:
        #print(trial.shape)
        channel_point_dict = get_top_k_channels_per_trial(trial, ch_names, channel_point_dict, k, update_func)
        channel_point_dicts_over_time.append(copy.deepcopy(channel_point_dict))
        #print(channel_point_dicts_over_time)
    
    top_k_channels = sorted(channel_point_dict, key=channel_point_dict.get, reverse=True)[:k]
    return top_k_channels, channel_point_dicts_over_time
    
def format_point_progression(point_progressions, ch_names):
    # this function collects the points over time per channel properly s.t. the point progression over time can easily be plotted
    point_progression_dict = {}
    for ch in ch_names:
        point_progression_dict[ch] = []


    for dict in point_progressions:
        for ch in ch_names:
            point_progression_dict[ch].append(dict[ch])

    return point_progression_dict



In [ ]:
def top_k_channel_per_trial_weighted(trial, ch_names,  channel_points_dict, k, update_func, aggregation_func=np.mean, **kwargs):
    # aggregate the importances award to each "pixel" in the trial over the time_point dimension according to the provided aggreagation_func
    #print(trial.shape)
    agg_points_per_channel = aggregation_func(trial, **kwargs)

    #print(agg_points_per_channel.shape)
    # get the indices of the top k channels
    top_k_channels_indices = np.argsort(agg_points_per_channel)[::-1][:k]
    top_k_channels_names = np.take(ch_names, top_k_channels_indices)
    #print(top_k_channels_names)
    # update the channels points according to the provided update func
    channel_points_dict = update_func(top_k_channels_names, channel_points_dict)
    return channel_points_dict

def get_top_k_weighted(explanations, ch_names, k, update_func, groupby_indices=np.array([])):
    #the previous top-k function considers each timepoint and each trial equally important.
    #in this version now each trial is still assumed to be equally important.
    #However, each timepoints importance is now weighted by the importance attributed to it by the attribution method.
    #we take the mean over the timepoint dimension. On one hand this increases susceptability to outliers, on the other hand it better allows
    #for some timepoints to be more important than others which I assume(for now) can absolutely be the case
    channel_point_dicts_over_time = []

    channel_point_dict = {}
    for ch in ch_names:
        channel_point_dict[ch] = 0

    if groupby_indices.any():
        data = explanations.squeeze()[groupby_indices]
    else:
        data = explanations.squeeze()
    # go through each trial in the dataset and assign points for each



    for trial in data:
        channel_point_dict = top_k_channel_per_trial_weighted(trial, ch_names, channel_point_dict, k, update_func, aggregation_func=np.mean, axis=1)
        channel_point_dicts_over_time.append(copy.deepcopy(channel_point_dict))

    top_k_channels = sorted(channel_point_dict, key=channel_point_dict.get, reverse=True)[:k]
    
    return top_k_channels, channel_point_dicts_over_time
    

def get_top_k_weighted_individual(explanations, ch_names, k, update_func, groupby_indices=np.array([])):
    # unlike the upper version which collects cumulative points per channel over trials
    # this function outputs the top-k chanels per trial for each trial individually
    channel_point_dicts_individual_trials = []

    if groupby_indices.any():
        data = explanations.squeeze()[groupby_indices]
    else:
        data = explanations.squeeze()

    for trial in data:
        channel_point_dict = {}
        for ch in ch_names:
            channel_point_dict[ch] = 0

        #print(trial.shape)
        channel_point_dict = top_k_channel_per_trial_weighted(trial, ch_names, channel_point_dict, k, update_func, axis=1)
        channel_point_dicts_individual_trials.append(copy.deepcopy(channel_point_dict))
        #print(channel_point_dicts_over_time)

    temp = format_point_progression(channel_point_dicts_individual_trials, ch_names)
    sum_over_channel_points_dict = {key: sum(value) for key,value in temp.items()}
    top_k_channels = sorted(sum_over_channel_points_dict, key=sum_over_channel_points_dict.get, reverse=True)[:k]

    return top_k_channels, channel_point_dicts_individual_trials


def top_k_groupby(aggregate_dict, aggregate_dict_key, groupby_dict, ch_names, k, update_func, top_k_func=get_top_k_weighted):
    groupby_channel_points_dict =  {}
    for groupby_key, groupby_indices in groupby_dict.items():
        groupby_channel_points_dict[groupby_key] = top_k_func(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices)
    
    return groupby_channel_points_dict
 

In [ ]:
cwd = os.getcwd()
cwd

In [ ]:
cwd = os.getcwd()
file_path = os.path.join(cwd, "../data/subject_007_preprocessed_combined_py.fif")
epochs = mne.read_epochs(file_path)
info = epochs.info
ch_names = epochs.ch_names

In [ ]:
def load_all_subject_results(subject_indices):
    results = {}
    for subject_index in subject_indices:
        with open(f'subject_{subject_index}_results.pkl', 'rb') as f:
            results[subject_index] = pickle.load(f)
    return results

# List of subject indices to load
cfg = load_config()
subject_indices = cfg.dataset.test_subject_indices

# Load results for each subject
subjects_dicts = load_all_subject_results(subject_indices)

In [ ]:
#for agg_func, agg_func_name in zip(agg_funcs, agg_funcs_names):
for i in range(len(subject_indices)):
    get_and_plot_topomap(np.abs(subjects_dicts[subject_indices[i]]["explanations"]), np.mean, info, aggregation_func_name="mean", save_path="./all_subjects/mean/abs", subject_idx=subject_indices[i], axis=(0,2))

In [ ]:
#for agg_func, agg_func_name in zip(agg_funcs, agg_funcs_names):
for i in range(len(subject_indices)):
    get_and_plot_topomap(subjects_dicts[subject_indices[i]]["explanations"], np.mean, info, aggregation_func_name="mean", save_path="./all_subjects/mean/normal", subject_idx=subject_indices[i], axis=(0,2))

In [ ]:
#for agg_func, agg_func_name in zip(agg_funcs, agg_funcs_names):
for i in range(len(subject_indices)):
    get_and_plot_topomap(subjects_dicts[subject_indices[i]]["explanations"], np.median, info, aggregation_func_name="median", save_path="./all_subjects/median/normal", subject_idx=subject_indices[i], axis=(0,2))

In [ ]:
#for agg_func, agg_func_name in zip(agg_funcs, agg_funcs_names):
for i in range(len(subject_indices)):
    get_and_plot_topomap(np.abs(subjects_dicts[subject_indices[i]]["explanations"]), np.median, info, aggregation_func_name="medoam", save_path="./all_subjects/median/normal", subject_idx=subject_indices[i], axis=(0,2))

In [ ]:
#for agg_func, agg_func_name in zip(agg_funcs, agg_funcs_names):

for i in range(len(subject_indices)):
    get_and_plot_topomap(np.abs(subjects_dicts[subject_indices[i]]["explanations"]), np.mean, info, aggregation_func_name="mean", save_path="./all_subjects/topoplots/DeepLift/", subject_idx=subject_indices[i], axis=(0,2))

In [ ]:
# get metric for comparing top-k channels
top_k_per_subject = []
for i in range(len(subject_indices)):
    top_k_linear_weighted, points_progression_linear_weighted = get_top_k_weighted(subjects_dicts[subject_indices[i]]["explanations"], ch_names, 10, update_channel_points_linear)
    top_k_per_subject.append((top_k_linear_weighted, points_progression_linear_weighted))

In [ ]:
def compute_common_channels(list1, list2):
    set1, set2 = set(list1), set(list2)
    return len(set1 & set2)

common_channels_matrix = np.zeros((len(subject_indices), len(subject_indices)))

for i in range(len(subject_indices)):
    for j in range(len(subject_indices)):
        common_channels_matrix[i, j] = compute_common_channels(top_k_per_subject[i][0], top_k_per_subject[j][0])

plt.figure(figsize=(10, 8))
seaborn.heatmap(common_channels_matrix, annot=True, cmap="viridis", xticklabels=subject_indices, yticklabels=subject_indices)
plt.title("Number of Common Channels in Top 10 for All Pairs of Subjects")
plt.xlabel("Subject Index")
plt.ylabel("Subject Index")
plt.show()

In [ ]:
# get metric for comparing top-k channels
top_k_per_subject_abs = []
for i in range(len(subject_indices)):
    top_k_linear_weighted, points_progression_linear_weighted = get_top_k_weighted(np.abs(subjects_dicts[subject_indices[i]]["explanations"]), ch_names, 10, update_channel_points_linear)
    top_k_per_subject_abs.append((top_k_linear_weighted, points_progression_linear_weighted))

In [ ]:
common_channels_matrix_abs = np.zeros((len(subject_indices), len(subject_indices)))

for i in range(len(subject_indices)):
    for j in range(len(subject_indices)):
        common_channels_matrix_abs[i, j] = compute_common_channels(top_k_per_subject_abs[i][0], top_k_per_subject_abs[j][0])

plt.figure(figsize=(10, 8))
seaborn.heatmap(common_channels_matrix_abs, annot=True, cmap="viridis", xticklabels=subject_indices, yticklabels=subject_indices)
plt.title("Number of Common Channels in Top 10 for All Pairs of Subjects")
plt.xlabel("Subject Index")
plt.ylabel("Subject Index")
plt.show()

# get agreement over time indices

In [ ]:
index_groups_all = []
for i in range(len(subject_indices)):
    index_groups_subject = []
    start = 0
    end = 0
    while end < len(subjects_dicts[subject_indices[i]]["uncertainties"]):
        if len(subjects_dicts[subject_indices[i]]["uncertainties"]) >= end + 100:
            end += 100
        else:
            end = len(subjects_dicts[subject_indices[i]]["uncertainties"])

        subjects_dicts[subject_indices[i]]["uncertainties"][start:end]
        index_group = np.zeros(len(subjects_dicts[subject_indices[i]]["uncertainties"]), dtype=bool)
        index_group[start:end] = True
        index_groups_subject.append(index_group)
        start += 100
    index_groups_all.append(np.array(index_groups_subject))

In [ ]:
subjects_dicts[subject_indices[i]]["explanations"][:100].shape

In [ ]:
# get metric for comparing top-k channels
top_k_per_subject_grouped = []
time_steps =np.arange(100,401,100)
for t in time_steps:
    subject_top_k_grouped = []
    for i in range(len(subject_indices)):
    
    
        if t == 100:
            top_k_linear_weighted, points_progression_linear_weighted = get_top_k_weighted(
            np.abs(subjects_dicts[subject_indices[i]]["explanations"][:t]), 
            ch_names, 
            10, 
            update_channel_points_linear
            )
            subject_top_k_grouped.append(top_k_linear_weighted)
        else:
            top_k_linear_weighted, points_progression_linear_weighted = get_top_k_weighted(
            np.abs(subjects_dicts[subject_indices[i]]["explanations"][t-100:t]), 
            ch_names, 
            10, 
            update_channel_points_linear
            )
            subject_top_k_grouped.append(top_k_linear_weighted)
            
    top_k_per_subject_grouped.append(subject_top_k_grouped)
           
      

In [ ]:
for idx, top_k_group in enumerate(top_k_per_subject_grouped):
    common_channels_matrix_grouped = np.zeros((len(subject_indices), len(subject_indices)))

    for i in range(len(subject_indices)):
        for j in range(len(subject_indices)):
            common_channels_matrix_grouped[i, j] = compute_common_channels(top_k_group[i], top_k_group[j])

    plt.figure(figsize=(10, 8))
    seaborn.heatmap(common_channels_matrix_grouped, annot=True, cmap="viridis", xticklabels=subject_indices, yticklabels=subject_indices)
    plt.title(f"Number of Common Channels in Top 10 for All Pairs of Subjects (Group {idx + 1})")
    plt.xlabel("Subject Index")
    plt.ylabel("Subject Index")
    plt.show()

    upper_tri_indices = np.triu_indices(len(subject_indices), k=1)
    common_channels_upper_tri = common_channels_matrix_grouped[upper_tri_indices]
    mean_common_channels = np.mean(common_channels_upper_tri)
    var_common_channels = np.std(common_channels_upper_tri)
    print(f"Group {idx + 1} - Mean of Common Channels: {mean_common_channels}, Variance of Common Channels: {var_common_channels}")

# heatmap to check channel importance development over time for each subject

In [ ]:
for i in range(len(subject_indices)):
    fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(15,5), sharey=True)
    fig.tight_layout()
    for j, groupby_key in enumerate(points_progression_per_subject_uncertainty[i].keys()):
        point_progression_dict = format_point_progression(points_progression_per_subject_uncertainty[i][groupby_key][1], ch_names)
        point_progression_df = pd.DataFrame.from_dict(point_progression_dict)
        fig.suptitle(f"subject: {subject_indices[i]}, groupby: uncertainty", y=-0.05)
        axs[j].set_title(groupby_key)
        
        hm = seaborn.heatmap(point_progression_df.transpose(), yticklabels=ch_names, ax=axs[j])
        if j ==0:
            hm.set_yticklabels(hm.get_yticklabels(), fontsize=5)
    dir_path = f"{cwd}/all_subjects/top_k/uncertainty/linear/heatmaps"
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)


    fig.savefig(dir_path+"/Subject_{subject_indices[i]}_uncertainty.png")




In [ ]:
points_progression_per_subject_uncertainty = []
for i in range(len(subject_indices)):
    points_progression_per_subject_uncertainty.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][0], ch_names, 10, update_channel_points_same, get_top_k_weighted_individual )))

In [ ]:
for i in range(len(subject_indices)):
    fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(15,5), sharey=True)
    fig.tight_layout()
    for j, groupby_key in enumerate(points_progression_per_subject_uncertainty[i].keys()):
        point_progression_dict = format_point_progression(points_progression_per_subject_uncertainty[i][groupby_key][1], ch_names)
        point_progression_df = pd.DataFrame.from_dict(point_progression_dict)
        fig.suptitle(f"subject: {subject_indices[i]}, groupby: uncertainty", y=-0.05)
        axs[j].set_title(groupby_key)
        
        hm = seaborn.heatmap(point_progression_df.transpose(), yticklabels=ch_names, ax=axs[j])
        if j ==0:
            hm.set_yticklabels(hm.get_yticklabels(), fontsize=5)
    fig.savefig(f"{cwd}/all_subjects/top_k/uncertainty/same/heatmaps/Subject_{subject_indices[i]}_uncertainty.png")


In [ ]:
points_progression_per_subject_pred_fixed = []
for i in range(len(subject_indices)):
    points_progression_per_subject_pred_fixed.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][1], ch_names, 10, update_channel_points_linear)))

In [ ]:
for i in range(len(subject_indices)):
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15,5), sharey=True)
    fig.tight_layout()
    for j, groupby_key in enumerate(points_progression_per_subject_pred_fixed[i].keys()):
        point_progression_dict = format_point_progression(points_progression_per_subject_pred_fixed[i][groupby_key][1], ch_names)
        point_progression_df = pd.DataFrame.from_dict(point_progression_dict)
        fig.suptitle(f"subject: {subject_indices[i]}, groupby: prediction fixed", y=-0.05)
        axs[j].set_title(groupby_key)
        
        hm = seaborn.heatmap(point_progression_df.transpose(), yticklabels=ch_names, ax=axs[j])
        if j ==0:
            hm.set_yticklabels(hm.get_yticklabels(), fontsize=5)
    fig.savefig(f"{cwd}/all_subjects/top_k/pred_binary_fixed/linear/heatmaps/Subject_{subject_indices[i]}.png")



In [ ]:
points_progression_per_subject_pred_fixed = []
for i in range(len(subject_indices)):
    points_progression_per_subject_pred_fixed.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][1], ch_names, 10, update_channel_points_same, get_top_k_weighted_individual)))

In [ ]:
for i in range(len(subject_indices)):
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15,5), sharey=True)
    fig.tight_layout()
    for j, groupby_key in enumerate(points_progression_per_subject_pred_fixed[i].keys()):
        point_progression_dict = format_point_progression(points_progression_per_subject_pred_fixed[i][groupby_key][1])
        point_progression_df = pd.DataFrame.from_dict(point_progression_dict)
        fig.suptitle(f"subject: {subject_indices[i]}, groupby: prediction fixed", y=-0.05)
        axs[j].set_title(groupby_key)
        
        hm = seaborn.heatmap(point_progression_df.transpose(), yticklabels=ch_names, ax=axs[j])
        if j ==0:
            hm.set_yticklabels(hm.get_yticklabels(), fontsize=5)
    fig.savefig(f"{cwd}/all_subjects/top_k/pred_binary_fixed/same/heatmaps/Subject_{subject_indices[i]}.png")

In [ ]:
points_progression_per_subject_pred_rolling = []
for i in range(len(subject_indices)):
    points_progression_per_subject_pred_rolling.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][2], ch_names, 10, update_channel_points_linear)))

In [ ]:
for i in range(len(subject_indices)):
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15,5), sharey=True)
    fig.tight_layout()
    for j, groupby_key in enumerate(points_progression_per_subject_pred_rolling[i].keys()):
        point_progression_dict = format_point_progression(points_progression_per_subject_pred_rolling[i][groupby_key][1], ch_names)
        point_progression_df = pd.DataFrame.from_dict(point_progression_dict)
        fig.suptitle(f"subject: {subject_indices[i]}, groupby: prediction rolling", y=-0.05)
        axs[j].set_title(groupby_key)
        
        hm = seaborn.heatmap(point_progression_df.transpose(), yticklabels=ch_names, ax=axs[j])
        if j ==0:
            hm.set_yticklabels(hm.get_yticklabels(), fontsize=5)
    fig.savefig(f"{cwd}/all_subjects/top_k/pred_binary_rolling/linear/heatmaps/Subject_{subject_indices[i]}.png")


In [ ]:
points_progression_per_subject_pred_rolling = []
for i in range(len(subject_indices)):
    points_progression_per_subject_pred_rolling.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][2], ch_names, 10, update_channel_points_same, get_top_k_weighted_individual)))

In [ ]:
for i in range(len(subject_indices)):
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15,5), sharey=True)
    fig.tight_layout()
    for j, groupby_key in enumerate(points_progression_per_subject_pred_rolling[i].keys()):
        point_progression_dict = format_point_progression(points_progression_per_subject_pred_rolling[i][groupby_key][1], ch_names)
        point_progression_df = pd.DataFrame.from_dict(point_progression_dict)
        fig.suptitle(f"subject: {subject_indices[i]}, groupby: prediction rolling", y=-0.05)
        axs[j].set_title(groupby_key)
        
        hm = seaborn.heatmap(point_progression_df.transpose(), yticklabels=ch_names, ax=axs[j])
        if j ==0:
            hm.set_yticklabels(hm.get_yticklabels(), fontsize=5)
    fig.savefig(f"{cwd}/all_subjects/top_k/pred_binary_rolling/same/heatmaps/Subject_{subject_indices[i]}.png")


In [ ]:
points_progression_per_subject_true = []
for i in range(len(subject_indices)):
    points_progression_per_subject_true.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][3], ch_names, 10, update_channel_points_linear)))

In [ ]:
for i in range(len(subject_indices)):
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15,5), sharey=True)
    fig.tight_layout()
    for j, groupby_key in enumerate(points_progression_per_subject_true[i].keys()):
        point_progression_dict = format_point_progression(points_progression_per_subject_true[i][groupby_key][1], ch_names)
        point_progression_df = pd.DataFrame.from_dict(point_progression_dict)
        fig.suptitle(f"subject: {subject_indices[i]}, groupby: prediction true", y=-0.05)
        axs[j].set_title(groupby_key)
        
        hm = seaborn.heatmap(point_progression_df.transpose(), yticklabels=ch_names, ax=axs[j])
        if j ==0:
            hm.set_yticklabels(hm.get_yticklabels(), fontsize=5)
    fig.savefig(f"{cwd}/all_subjects/top_k/true_binary/linear/heatmaps/Subject_{subject_indices[i]}.png")

In [ ]:
points_progression_per_subject_true = []
for i in range(len(subject_indices)):
    points_progression_per_subject_true.append((top_k_groupby(subjects_dicts[i], "dl_agg", groupby_dicts_list[i][3], ch_names, 10, update_channel_points_same, get_top_k_weighted_individual)))

In [ ]:
for i in range(len(subject_indices)):
    fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(15,5), sharey=True)
    fig.tight_layout()
    for j, groupby_key in enumerate(points_progression_per_subject_true[i].keys()):
        point_progression_dict = format_point_progression(points_progression_per_subject_true[i][groupby_key][1], ch_names)
        point_progression_df = pd.DataFrame.from_dict(point_progression_dict)
        fig.suptitle(f"subject: {subject_indices[i]}, groupby: prediction true", y=-0.05)
        axs[j].set_title(groupby_key)
        
        hm = seaborn.heatmap(point_progression_df.transpose(), yticklabels=ch_names, ax=axs[j])
        if j ==0:
            hm.set_yticklabels(hm.get_yticklabels(), fontsize=5)
    fig.savefig(f"{cwd}/all_subjects/top_k/true_binary/same/heatmaps/Subject_{subject_indices[i]}.png")